In [1]:
import json

class ACLMessage:
    def __init__(self, performative):
        self.performative = performative   # REQUEST, INFORM, AGREE, REFUSE など
        self.sender = ""
        self.receivers = []
        self.content = ""
        self.language = ""
        self.ontology = ""
        self.conversation_id = ""
        self.reply_with = ""
        self.in_reply_to = ""
        self.reply_by = ""

def send_message(msg: ACLMessage):
    print("[SEND]")
    print(f" performative : {msg.performative}")
    print(f" sender       : {msg.sender}")
    print(f" receivers    : {msg.receivers}")
    print(f" language     : {msg.language}")
    print(f" ontology     : {msg.ontology}")
    print(f" conversation : {msg.conversation_id}")
    print(" content (JSON) :")
    print(f" {msg.content}")
    print("--------------")

def receive_message(
    mock_performative,
    mock_content,
    sender,
    receivers,
    conv_id
):
    """擬似的に受信メッセージを生成し返す関数。"""
    msg = ACLMessage(mock_performative)
    msg.sender = sender
    msg.receivers = receivers
    msg.content = mock_content
    msg.language = "application/json"     # JSON形式でやりとりしていると明示
    msg.ontology = "InventoryOntology"    # 在庫管理関連のオントロジー
    msg.conversation_id = conv_id
    return msg

def agentA_behavior():
    """AgentA（発注担当）の疑似的な振る舞い。"""
    conv_id = "order-001"

    # (1) 在庫確認のREQUESTメッセージを作成
    check_stock_content = {
        "action": "CHECK_STOCK",
        "product": {
            "id": "XYZ",
            "name": "商品XYZ"
        },
        "requestedInfo": ["currentStock", "price"]
    }
    msg_request_stock = ACLMessage("REQUEST")
    msg_request_stock.sender = "AgentA"
    msg_request_stock.receivers = ["AgentB"]
    msg_request_stock.language = "application/json"
    msg_request_stock.ontology = "InventoryOntology"
    msg_request_stock.conversation_id = conv_id
    msg_request_stock.content = json.dumps(check_stock_content, ensure_ascii=False)

    send_message(msg_request_stock)

    # (2) AgentBからの在庫情報（INFORM）を受け取る（擬似）
    received_stock_info = receive_message(
        mock_performative="INFORM",
        mock_content=json.dumps({
            "response": "STOCK_INFO",
            "product": {
                "id": "XYZ",
                "name": "商品XYZ"
            },
            "available": 30,
            "unitPrice": 1200
        }, ensure_ascii=False),
        sender="AgentB",
        receivers=["AgentA"],
        conv_id=conv_id
    )

    print("[RECEIVE]")
    print(f" performative : {received_stock_info.performative}")
    print(f" sender       : {received_stock_info.sender}")
    print(" content (JSON) :")
    print(received_stock_info.content)
    print("--------------")

    # 在庫情報を解析（JSONをパース）
    stock_info = json.loads(received_stock_info.content)
    available_stock = stock_info["available"]

    # (3) 在庫数が十分なので、ORDERのREQUESTを送る
    if available_stock >= 10:
        order_content = {
            "action": "ORDER",
            "orderDetails": {
                "productId": "XYZ",
                "quantity": 10
            },
            "expectedDeliveryDate": "2025-06-30"
        }
        msg_order = ACLMessage("REQUEST")
        msg_order.sender = "AgentA"
        msg_order.receivers = ["AgentB"]
        msg_order.language = "application/json"
        msg_order.ontology = "InventoryOntology"
        msg_order.conversation_id = conv_id
        msg_order.content = json.dumps(order_content, ensure_ascii=False)

        send_message(msg_order)

        # (4) AgentBが注文を受け入れるかどうか返す（AGREE or REFUSE）
        received_order_reply = receive_message(
            mock_performative="AGREE",  # 例としてAGREEを返す
            mock_content=json.dumps({
                "response": "ORDER_ACCEPTED",
                "expectedDeliveryDate": "2025-06-30"
            }, ensure_ascii=False),
            sender="AgentB",
            receivers=["AgentA"],
            conv_id=conv_id
        )
        print("[RECEIVE]")
        print(f" performative : {received_order_reply.performative}")
        print(f" sender       : {received_order_reply.sender}")
        print(" content (JSON) :")
        print(received_order_reply.content)
        print("--------------")

        # (5) 納品完了後、AgentBがINFORMで通知
        received_delivery_info = receive_message(
            mock_performative="INFORM",
            mock_content=json.dumps({
                "response": "DELIVERY_DONE",
                "productId": "XYZ",
                "deliveredQuantity": 10
            }, ensure_ascii=False),
            sender="AgentB",
            receivers=["AgentA"],
            conv_id=conv_id
        )
        print("[RECEIVE]")
        print(f" performative : {received_delivery_info.performative}")
        print(f" sender       : {received_delivery_info.sender}")
        print(" content (JSON) :")
        print(received_delivery_info.content)
        print("--------------")
    else:
        print("在庫が不足しているため、注文を行いません。")


if __name__ == "__main__":
    agentA_behavior()


[SEND]
 performative : REQUEST
 sender       : AgentA
 receivers    : ['AgentB']
 language     : application/json
 ontology     : InventoryOntology
 conversation : order-001
 content (JSON) :
 {"action": "CHECK_STOCK", "product": {"id": "XYZ", "name": "商品XYZ"}, "requestedInfo": ["currentStock", "price"]}
--------------
[RECEIVE]
 performative : INFORM
 sender       : AgentB
 content (JSON) :
{"response": "STOCK_INFO", "product": {"id": "XYZ", "name": "商品XYZ"}, "available": 30, "unitPrice": 1200}
--------------
[SEND]
 performative : REQUEST
 sender       : AgentA
 receivers    : ['AgentB']
 language     : application/json
 ontology     : InventoryOntology
 conversation : order-001
 content (JSON) :
 {"action": "ORDER", "orderDetails": {"productId": "XYZ", "quantity": 10}, "expectedDeliveryDate": "2025-06-30"}
--------------
[RECEIVE]
 performative : AGREE
 sender       : AgentB
 content (JSON) :
{"response": "ORDER_ACCEPTED", "expectedDeliveryDate": "2025-06-30"}
--------------
[RECEIV